# DecodeLabs Data Science Internship - Project 1
## Advanced EDA & Feature Engineering

**Author:** Muhammad Usaid  
**Objective:** Transform raw e-commerce order data into a mathematically clean, machine-learning-ready dataset using statistical missing-data treatment, IQR outlier handling, vectorized transformations, categorical encoding, feature engineering, correlation auditing, and structural validation.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OneHotEncoder

ROOT = Path("..")
RAW = ROOT / "data" / "raw" / "Dataset_for_Data_Analytics.xlsx"
df = pd.read_excel(RAW)
df["Date"] = pd.to_datetime(df["Date"])
df.head()


## 1. Input Fidelity: shape, schema, uniqueness, missingness

In [ ]:
print("Shape:", df.shape)
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Duplicate OrderID:", df["OrderID"].duplicated().sum())
print("Duplicate TrackingNumber:", df["TrackingNumber"].duplicated().sum())

profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100).round(2),
    "unique": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)
profile


### Missing-data decision
The source dataset has no numeric missing values. `CouponCode` is the only incomplete feature (309 missing / 25.75%). A blank coupon is a meaningful business state, so the applied treatment is `NO_COUPON`. Mean, median, and KNN numeric-imputation functions are still implemented below to satisfy the reusable pipeline requirement without fabricating missingness in already-complete numeric columns.


In [ ]:
NUMERIC = ["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]

def impute_numeric(frame, strategy="median"):
    out = frame.copy()
    cols = [c for c in NUMERIC if out[c].isna().any()]
    if not cols:
        return out
    if strategy == "mean":
        out[cols] = out[cols].fillna(out[cols].mean())
    elif strategy == "median":
        out[cols] = out[cols].fillna(out[cols].median())
    elif strategy == "knn":
        out[cols] = KNNImputer(n_neighbors=5).fit_transform(out[cols])
    else:
        raise ValueError("Use mean, median, or knn")
    return out

clean = impute_numeric(df, "median")
clean["CouponCode_Imputed"] = clean["CouponCode"].fillna("NO_COUPON")
clean["CouponCode_Imputed"].value_counts(dropna=False)


## 2. Outlier Isolation with IQR

In [ ]:
def iqr_audit(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    mask = ~series.between(lo, hi)
    return q1, q3, lo, hi, mask

rows = []
for col in NUMERIC:
    q1, q3, lo, hi, mask = iqr_audit(clean[col])
    rows.append([col, q1, q3, lo, hi, int(mask.sum())])
pd.DataFrame(rows, columns=["feature","Q1","Q3","lower","upper","outliers"])


In [ ]:
q1, q3, lower, upper, mask = iqr_audit(clean["TotalPrice"])
clean["TotalPrice_IQR_Outlier"] = mask.astype("int8")
clean["TotalPrice_Winsorized"] = clean["TotalPrice"].clip(lower, upper)

print("IQR bounds:", round(lower, 2), round(upper, 2))
print("Flagged orders:", int(mask.sum()))
clean.loc[mask, ["OrderID","Product","Quantity","UnitPrice","TotalPrice","TotalPrice_Winsorized"]]


The flagged high-value orders are mathematically extreme relative to the distribution but remain internally valid because `TotalPrice = Quantity × UnitPrice`. Therefore the source amount is preserved and a separate winsorized modeling feature is created instead of deleting rows.


## 3. Vectorized Feature Engineering

In [ ]:
clean["HasCoupon"] = clean["CouponCode_Imputed"].ne("NO_COUPON").astype("int8")
clean["OrderYear"] = clean["Date"].dt.year
clean["OrderMonth"] = clean["Date"].dt.month
clean["OrderQuarter"] = clean["Date"].dt.quarter
clean["OrderDayOfWeek"] = clean["Date"].dt.day_name()
clean["IsWeekend"] = clean["Date"].dt.dayofweek.ge(5).astype("int8")
clean["BasketFillRatio"] = clean["Quantity"].div(clean["ItemsInCart"])
clean["OrderValuePerCartItem"] = clean["TotalPrice"].div(clean["ItemsInCart"])

engineered = [
    "CouponCode_Imputed","HasCoupon","OrderYear","OrderMonth","OrderQuarter",
    "OrderDayOfWeek","IsWeekend","BasketFillRatio","OrderValuePerCartItem",
    "TotalPrice_IQR_Outlier","TotalPrice_Winsorized"
]
clean[engineered].head()


## 4. Categorical Translation: One-Hot Encoding

In [ ]:
categorical = ["Product", "PaymentMethod", "OrderStatus", "ReferralSource"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.int8)
matrix = encoder.fit_transform(clean[categorical])
encoded = pd.DataFrame(matrix, columns=encoder.get_feature_names_out(categorical), index=clean.index)
model_frame = pd.concat([clean, encoded], axis=1)
encoded.head()


## 5. Multicollinearity Audit

In [ ]:
corr = model_frame.select_dtypes(include=np.number).corr().abs()
upper_triangle = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = (
    upper_triangle.stack()
    .rename("abs_correlation")
    .reset_index()
    .rename(columns={"level_0":"feature_a","level_1":"feature_b"})
)
pairs.query("abs_correlation > 0.80").sort_values("abs_correlation", ascending=False).head(20)


Do not automatically drop the first correlated variable found. Once a modeling target is selected, compare each member of a highly correlated pair against that target and retain the more informative or more stable feature.


## 6. Structural Contracts / Business Validation

In [ ]:
assert clean["OrderID"].is_unique
assert clean["TrackingNumber"].is_unique
assert clean["Quantity"].between(1, 5).all()
assert clean["ItemsInCart"].gt(0).all()
assert clean["UnitPrice"].gt(0).all()
assert clean["TotalPrice"].gt(0).all()
assert np.allclose(clean["TotalPrice"], clean["Quantity"] * clean["UnitPrice"], atol=0.01)
print("All structural and business-rule checks passed.")


## 7. EDA Visuals

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
df.isna().sum().plot(kind="bar", ax=ax)
ax.set_title("Missing Values by Feature")
ax.set_ylabel("Count")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.boxplot(clean["TotalPrice"], vert=False)
ax.set_title("TotalPrice IQR Inspection")
ax.set_xlabel("TotalPrice")
plt.tight_layout()
plt.show()


## 8. Business-Level Summary

In [ ]:
summary = {
    "rows": len(clean),
    "total_revenue": clean["TotalPrice"].sum(),
    "average_order_value": clean["TotalPrice"].mean(),
    "coupon_usage_pct": clean["HasCoupon"].mean() * 100,
    "iqr_outliers": clean["TotalPrice_IQR_Outlier"].sum(),
    "top_product_by_revenue": clean.groupby("Product")["TotalPrice"].sum().idxmax(),
}
summary


## 9. Output & Production Extension

The cleaned feature set is written to `data/processed/cleaned_orders_features.csv`. The repository also includes an optional Pandera schema contract. A feature store such as Feast would be appropriate only when these features must be served consistently to both offline training and real-time inference. For historical training, feature joins must be point-in-time correct so no future information leaks into past examples.


In [ ]:
OUT = ROOT / "data" / "processed" / "cleaned_orders_features_from_notebook.csv"
clean.to_csv(OUT, index=False)
print("Saved:", OUT)
